In [ ]:
# To install dependencies: pip install -r requirements.txt

In [ ]:
from pathlib import Path
import os

import overpy
import geopandas as gpd
from shapely.geometry import Point, box, Polygon

# Ensure working directory is the repository root (not notebooks/)
if Path.cwd().name == "notebooks":
    os.chdir("..")

**Introduce City Shapefile**

In [ ]:
# Read study-area boundary shapefile
boundary_path = Path("data") / "Mask.shp"
boundary = gpd.read_file(boundary_path)

# Convert to WGS84 (required by the Overpass API)
boundary = boundary.to_crs("EPSG:4326")

# Build bounding box from boundary
minx, miny, maxx, maxy = boundary.total_bounds  # boundary bounds

**Output Folder**

In [ ]:
# Output directory
output_folder = Path("outputs")
output_folder.mkdir(parents=True, exist_ok=True)

**H1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="hospital"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="hospital"]({miny},{minx},{maxy},{maxx});
  node["amenity"="clinic"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="clinic"]({miny},{minx},{maxy},{maxx});
  node["amenity"="doctors"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="doctor"]({miny},{minx},{maxy},{maxx});
  node["amenity"="dentist"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="dentist"]({miny},{minx},{maxy},{maxx});
  node["healthcare:speciality"="orthodontics"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="physiotherapist"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="hospital"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="hospital"]({miny},{minx},{maxy},{maxx});
  way["amenity"="clinic"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="clinic"]({miny},{minx},{maxy},{maxx});
  way["amenity"="doctors"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="doctor"]({miny},{minx},{maxy},{maxx});
  way["amenity"="dentist"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="dentist"]({miny},{minx},{maxy},{maxx});
  way["healthcare:speciality"="orthodontics"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="physiotherapist"]({miny},{minx},{maxy},{maxx});

  relation["amenity"="hospital"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="hospital"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="clinic"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="clinic"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="doctors"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="doctor"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="dentist"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="dentist"]({miny},{minx},{maxy},{maxx});
  relation["healthcare:speciality"="orthodontics"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="physiotherapist"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    # Use node coordinates directly
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "healthcare": node.tags.get("healthcare"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "healthcare": way.tags.get("healthcare"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "healthcare": relation.tags.get("healthcare"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "H1"

# Save as a Shapefile
layer_name = "H1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**H2 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="pharmacy"]({miny},{minx},{maxy},{maxx});
  node["healthcare"="pharmacy"]({miny},{minx},{maxy},{maxx});
  way["amenity"="pharmacy"]({miny},{minx},{maxy},{maxx});
  way["healthcare"="pharmacy"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="pharmacy"]({miny},{minx},{maxy},{maxx});
  relation["healthcare"="pharmacy"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    # Use node coordinates directly
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "healthcare": node.tags.get("healthcare"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "healthcare": way.tags.get("healthcare"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "healthcare": relation.tags.get("healthcare"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "H2"

# Save as a Shapefile
layer_name = "H2.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**H3 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["social_facility"="assisted_living"]({miny},{minx},{maxy},{maxx});
  node["social_facility"="group_home"]({miny},{minx},{maxy},{maxx});
  node["social_facility:for"="senior"]({miny},{minx},{maxy},{maxx});
  node["social_facility"="nursing_home"]({miny},{minx},{maxy},{maxx});
  node["amenity"="nursing_home"]({miny},{minx},{maxy},{maxx});

  way["social_facility"="assisted_living"]({miny},{minx},{maxy},{maxx});
  way["social_facility"="group_home"]({miny},{minx},{maxy},{maxx});
  way["social_facility:for"="senior"]({miny},{minx},{maxy},{maxx});
  way["social_facility"="nursing_home"]({miny},{minx},{maxy},{maxx});
  way["amenity"="nursing_home"]({miny},{minx},{maxy},{maxx});

  relation["social_facility"="assisted_living"]({miny},{minx},{maxy},{maxx});
  relation["social_facility"="group_home"]({miny},{minx},{maxy},{maxx});
  relation["social_facility:for"="senior"]({miny},{minx},{maxy},{maxx});
  relation["social_facility"="nursing_home"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="nursing_home"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    # Use node coordinates directly
    data.append({
        "id": node.id,
        "social_fac": node.tags.get("social_facility"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "social_fac": way.tags.get("social_facility"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "social_fac": relation.tags.get("social_facility"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "H3"

# Save as a Shapefile
layer_name = "H3.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**E Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="school"]({miny},{minx},{maxy},{maxx});
  node["amenity"="kindergarten"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="school"]({miny},{minx},{maxy},{maxx});
  way["amenity"="kindergarten"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="school"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="kindergarten"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    # Use node coordinates directly
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Drop duplicates by 'name' (keep first); ignore null names
gdf = gdf[gdf['name'].isna() | ~gdf.duplicated(subset='name', keep='first')]

# Add Category column
gdf["Category"] = "E"

# Save as a Shapefile
layer_name = "E.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**T Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["railway"="station"]({miny},{minx},{maxy},{maxx});
  node["public_transport"="stop_position"]({miny},{minx},{maxy},{maxx});
  node["public_transport"="platform"]({miny},{minx},{maxy},{maxx});
  node["railway"="subway_entrance"]({miny},{minx},{maxy},{maxx});
  node["railway"="tram_stop"]({miny},{minx},{maxy},{maxx});
  node["railway"="platform"]({miny},{minx},{maxy},{maxx});
  node["amenity"="bus_station"]({miny},{minx},{maxy},{maxx});
  node["highway"="bus_stop"]({miny},{minx},{maxy},{maxx});
  node["highway"="platform"]({miny},{minx},{maxy},{maxx});
  
  way["railway"="station"]({miny},{minx},{maxy},{maxx});
  way["public_transport"="stop_position"]({miny},{minx},{maxy},{maxx});
  way["public_transport"="platform"]({miny},{minx},{maxy},{maxx});
  way["railway"="subway_entrance"]({miny},{minx},{maxy},{maxx});
  way["railway"="tram_stop"]({miny},{minx},{maxy},{maxx});
  way["railway"="platform"]({miny},{minx},{maxy},{maxx});
  way["amenity"="bus_station"]({miny},{minx},{maxy},{maxx});
  way["highway"="bus_stop"]({miny},{minx},{maxy},{maxx});
  way["highway"="platform"]({miny},{minx},{maxy},{maxx});
  
  relation["railway"="station"]({miny},{minx},{maxy},{maxx});
  relation["public_transport"="stop_position"]({miny},{minx},{maxy},{maxx});
  relation["public_transport"="platform"]({miny},{minx},{maxy},{maxx});
  relation["railway"="subway_entrance"]({miny},{minx},{maxy},{maxx});
  relation["railway"="tram_stop"]({miny},{minx},{maxy},{maxx});
  relation["railway"="platform"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="bus_station"]({miny},{minx},{maxy},{maxx});
  relation["highway"="bus_stop"]({miny},{minx},{maxy},{maxx});
  relation["highway"="platform"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    # Use node coordinates directly
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "railway": node.tags.get("railway"),
        "public_tra": node.tags.get("public_transport"),
        "highway": node.tags.get("highway"),
        "bus": node.tags.get("bus"),
        "train": node.tags.get("train"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "railway": way.tags.get("railway"),
            "public_tra": way.tags.get("public_transport"),
            "highway": way.tags.get("highway"),
            "bus": way.tags.get("bus"),
            "train": way.tags.get("train"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "railway": relation.tags.get("railway"),
            "public_tra": relation.tags.get("public_transport"),
            "highway": relation.tags.get("highway"),
            "bus": relation.tags.get("bus"),
            "train": relation.tags.get("train"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "T"

# Save as a Shapefile
layer_name = "T.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**G1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="bank"]({miny},{minx},{maxy},{maxx});
  node["amenity"="atm"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="bank"]({miny},{minx},{maxy},{maxx});
  way["amenity"="atm"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="bank"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="atm"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "G1"

# Save as a Shapefile
layer_name = "G1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**G2 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="post_office"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="post_office"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="post_office"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "G2"

# Save as a Shapefile
layer_name = "G2.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**C1 Category**

In [ ]:

# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="restaurant"]({miny},{minx},{maxy},{maxx});
  node["amenity"="fast_food"]({miny},{minx},{maxy},{maxx});
  node["amenity"="food_court"]({miny},{minx},{maxy},{maxx});
  node["amenity"="pub"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="restaurant"]({miny},{minx},{maxy},{maxx});
  way["amenity"="fast_food"]({miny},{minx},{maxy},{maxx});
  way["amenity"="food_court"]({miny},{minx},{maxy},{maxx});
  way["amenity"="pub"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="restaurant"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="fast_food"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="food_court"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="pub"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "C1"

# Save as a Shapefile
layer_name = "C1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**C2_1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  node["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  node["shop"="general"]({miny},{minx},{maxy},{maxx});
  
  way["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  way["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  way["shop"="general"]({miny},{minx},{maxy},{maxx});
  
  relation["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  relation["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  relation["shop"="general"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "shop": node.tags.get("shop"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "shop": way.tags.get("shop"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "shop": relation.tags.get("shop"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "C2_1"

# Save as a Shapefile
layer_name = "C2_1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**C2_2 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  node["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  node["shop"="general"]({miny},{minx},{maxy},{maxx});
  node["shop"="kiosk"]({miny},{minx},{maxy},{maxx});
  node["shop"="bakery"]({miny},{minx},{maxy},{maxx});
  node["shop"="butcher"]({miny},{minx},{maxy},{maxx});
  node["shop"="seafood"]({miny},{minx},{maxy},{maxx});
  node["shop"="dairy"]({miny},{minx},{maxy},{maxx});
  node["shop"="cheese"]({miny},{minx},{maxy},{maxx});
  node["shop"="frozen_food"]({miny},{minx},{maxy},{maxx});
  node["shop"="deli"]({miny},{minx},{maxy},{maxx});
  node["shop"="pastry"]({miny},{minx},{maxy},{maxx});
  node["craft"="confectionery"]({miny},{minx},{maxy},{maxx});
  node["shop"="chocolate"]({miny},{minx},{maxy},{maxx});
  node["shop"="tea"]({miny},{minx},{maxy},{maxx});
  node["shop"="coffee"]({miny},{minx},{maxy},{maxx});
  node["shop"="herbalist"]({miny},{minx},{maxy},{maxx});
  node["amenity"="marketplace"]({miny},{minx},{maxy},{maxx});
  node["shop"="greengrocer"]({miny},{minx},{maxy},{maxx});
  node["shop"="farm"]({miny},{minx},{maxy},{maxx});
  node["shop"="organic"]({miny},{minx},{maxy},{maxx});
  node["shop"="water"]({miny},{minx},{maxy},{maxx});
  node["shop"="alcohol"]({miny},{minx},{maxy},{maxx});
  node["shop"="beverages"]({miny},{minx},{maxy},{maxx});
  node["shop"="wine"]({miny},{minx},{maxy},{maxx});
  node["shop"="cannabis"]({miny},{minx},{maxy},{maxx});
  node["shop"="nutrition_supplements"]({miny},{minx},{maxy},{maxx});
  node["shop"="health_food"]({miny},{minx},{maxy},{maxx});
  node["shop"="spices"]({miny},{minx},{maxy},{maxx});
  
  way["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  way["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  way["shop"="general"]({miny},{minx},{maxy},{maxx});
  way["shop"="kiosk"]({miny},{minx},{maxy},{maxx});
  way["shop"="bakery"]({miny},{minx},{maxy},{maxx});
  way["shop"="butcher"]({miny},{minx},{maxy},{maxx});
  way["shop"="seafood"]({miny},{minx},{maxy},{maxx});
  way["shop"="dairy"]({miny},{minx},{maxy},{maxx});
  way["shop"="cheese"]({miny},{minx},{maxy},{maxx});
  way["shop"="frozen_food"]({miny},{minx},{maxy},{maxx});
  way["shop"="deli"]({miny},{minx},{maxy},{maxx});
  way["shop"="pastry"]({miny},{minx},{maxy},{maxx});
  way["craft"="confectionery"]({miny},{minx},{maxy},{maxx});
  way["shop"="chocolate"]({miny},{minx},{maxy},{maxx});
  way["shop"="tea"]({miny},{minx},{maxy},{maxx});
  way["shop"="coffee"]({miny},{minx},{maxy},{maxx});
  way["shop"="herbalist"]({miny},{minx},{maxy},{maxx});
  way["amenity"="marketplace"]({miny},{minx},{maxy},{maxx});
  way["shop"="greengrocer"]({miny},{minx},{maxy},{maxx});
  way["shop"="farm"]({miny},{minx},{maxy},{maxx});
  way["shop"="organic"]({miny},{minx},{maxy},{maxx});
  way["shop"="water"]({miny},{minx},{maxy},{maxx});
  way["shop"="alcohol"]({miny},{minx},{maxy},{maxx});
  way["shop"="beverages"]({miny},{minx},{maxy},{maxx});
  way["shop"="wine"]({miny},{minx},{maxy},{maxx});
  way["shop"="cannabis"]({miny},{minx},{maxy},{maxx});
  way["shop"="nutrition_supplements"]({miny},{minx},{maxy},{maxx});
  way["shop"="health_food"]({miny},{minx},{maxy},{maxx});
  way["shop"="spices"]({miny},{minx},{maxy},{maxx});
  
  relation["shop"="supermarket"]({miny},{minx},{maxy},{maxx});
  relation["shop"="convenience"]({miny},{minx},{maxy},{maxx});
  relation["shop"="general"]({miny},{minx},{maxy},{maxx});
  relation["shop"="kiosk"]({miny},{minx},{maxy},{maxx});
  relation["shop"="bakery"]({miny},{minx},{maxy},{maxx});
  relation["shop"="butcher"]({miny},{minx},{maxy},{maxx});
  relation["shop"="seafood"]({miny},{minx},{maxy},{maxx});
  relation["shop"="dairy"]({miny},{minx},{maxy},{maxx});
  relation["shop"="cheese"]({miny},{minx},{maxy},{maxx});
  relation["shop"="frozen_food"]({miny},{minx},{maxy},{maxx});
  relation["shop"="deli"]({miny},{minx},{maxy},{maxx});
  relation["shop"="pastry"]({miny},{minx},{maxy},{maxx});
  relation["craft"="confectionery"]({miny},{minx},{maxy},{maxx});
  relation["shop"="chocolate"]({miny},{minx},{maxy},{maxx});
  relation["shop"="tea"]({miny},{minx},{maxy},{maxx});
  relation["shop"="coffee"]({miny},{minx},{maxy},{maxx});
  relation["shop"="herbalist"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="marketplace"]({miny},{minx},{maxy},{maxx});
  relation["shop"="greengrocer"]({miny},{minx},{maxy},{maxx});
  relation["shop"="farm"]({miny},{minx},{maxy},{maxx});
  relation["shop"="organic"]({miny},{minx},{maxy},{maxx});
  relation["shop"="water"]({miny},{minx},{maxy},{maxx});
  relation["shop"="alcohol"]({miny},{minx},{maxy},{maxx});
  relation["shop"="beverages"]({miny},{minx},{maxy},{maxx});
  relation["shop"="wine"]({miny},{minx},{maxy},{maxx});
  relation["shop"="cannabis"]({miny},{minx},{maxy},{maxx});
  relation["shop"="nutrition_supplements"]({miny},{minx},{maxy},{maxx});
  relation["shop"="health_food"]({miny},{minx},{maxy},{maxx});
  relation["shop"="spices"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "shop": node.tags.get("shop"),
        "craft": node.tags.get("craft"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "shop": way.tags.get("shop"),
            "craft": way.tags.get("craft"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "shop": relation.tags.get("shop"),
            "craft": relation.tags.get("craft"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "C2_2"

# Save as a Shapefile
layer_name = "C2_2.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**C3_1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
  
  way["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
  
  relation["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "shop": node.tags.get("shop"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "shop": way.tags.get("shop"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "shop": relation.tags.get("shop"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "C3_1"

# Save as a Shapefile
layer_name = "C3_1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**C3_2 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["shop"="clothes"]({miny},{minx},{maxy},{maxx});
  node["shop"="boutique"]({miny},{minx},{maxy},{maxx});
  node["shop"="shoes"]({miny},{minx},{maxy},{maxx});
  node["shop"="dry_cleaning"]({miny},{minx},{maxy},{maxx});
  node["shop"="laundry"]({miny},{minx},{maxy},{maxx});
  node["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
  node["shop"="beauty"]({miny},{minx},{maxy},{maxx});
  
  way["shop"="clothes"]({miny},{minx},{maxy},{maxx});
  way["shop"="boutique"]({miny},{minx},{maxy},{maxx});
  way["shop"="shoes"]({miny},{minx},{maxy},{maxx});
  way["shop"="dry_cleaning"]({miny},{minx},{maxy},{maxx});
  way["shop"="laundry"]({miny},{minx},{maxy},{maxx});
  way["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
  way["shop"="beauty"]({miny},{minx},{maxy},{maxx});
  
  relation["shop"="clothes"]({miny},{minx},{maxy},{maxx});
  relation["shop"="boutique"]({miny},{minx},{maxy},{maxx});
  relation["shop"="shoes"]({miny},{minx},{maxy},{maxx});
  relation["shop"="dry_cleaning"]({miny},{minx},{maxy},{maxx});
  relation["shop"="laundry"]({miny},{minx},{maxy},{maxx});
  relation["shop"="hairdresser"]({miny},{minx},{maxy},{maxx});
  relation["shop"="beauty"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "shop": node.tags.get("shop"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "shop": way.tags.get("shop"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "shop": relation.tags.get("shop"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "C3_2"

# Save as a Shapefile
layer_name = "C3_2.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**En1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["tourism"="museum"]({miny},{minx},{maxy},{maxx});
  node["tourism"="gallery"]({miny},{minx},{maxy},{maxx});
  node["amenity"="theatre"]({miny},{minx},{maxy},{maxx});
  node["amenity"="library"]({miny},{minx},{maxy},{maxx});
  node["amenity"="planetarium"]({miny},{minx},{maxy},{maxx});
  node["amenity"="arts_centre"]({miny},{minx},{maxy},{maxx});
  node["amenity"="community_centre"]({miny},{minx},{maxy},{maxx});
  node["amenity"="conference_centre"]({miny},{minx},{maxy},{maxx});
  
  way["tourism"="museum"]({miny},{minx},{maxy},{maxx});
  way["tourism"="gallery"]({miny},{minx},{maxy},{maxx});
  way["amenity"="theatre"]({miny},{minx},{maxy},{maxx});
  way["amenity"="library"]({miny},{minx},{maxy},{maxx});
  way["amenity"="planetarium"]({miny},{minx},{maxy},{maxx});
  way["amenity"="arts_centre"]({miny},{minx},{maxy},{maxx});
  way["amenity"="community_centre"]({miny},{minx},{maxy},{maxx});
  way["amenity"="conference_centre"]({miny},{minx},{maxy},{maxx});
  
  relation["tourism"="museum"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="gallery"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="theatre"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="library"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="planetarium"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="arts_centre"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="community_centre"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="conference_centre"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "tourism": node.tags.get("tourism"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "tourism": way.tags.get("tourism"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "tourism": relation.tags.get("tourism"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "En1"

# Save as a Shapefile
layer_name = "En1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**En2_1 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["leisure"="playground"]({miny},{minx},{maxy},{maxx});
  node["playground"="slide"]({miny},{minx},{maxy},{maxx});
  node["playground"="swing"]({miny},{minx},{maxy},{maxx});
  node["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  node["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  node["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  node["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  way["leisure"="playground"]({miny},{minx},{maxy},{maxx});
  way["playground"="slide"]({miny},{minx},{maxy},{maxx});
  way["playground"="swing"]({miny},{minx},{maxy},{maxx});
  way["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  way["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  way["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  way["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  relation["leisure"="playground"]({miny},{minx},{maxy},{maxx});
  relation["playground"="slide"]({miny},{minx},{maxy},{maxx});
  relation["playground"="swing"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="park"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "leisure": node.tags.get("leisure"),
        "playground": node.tags.get("playground"),
        "tourism": node.tags.get("tourism"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "leisure": way.tags.get("leisure"),
            "playground": way.tags.get("playground"),
            "tourism": way.tags.get("tourism"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "leisure": relation.tags.get("leisure"),
            "playground": relation.tags.get("playground"),
            "tourism": relation.tags.get("tourism"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "En2_1"

# Save as a Shapefile
layer_name = "En2_1.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

**En2_2 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  node["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  node["social_facility"="social_club"]({miny},{minx},{maxy},{maxx});
  node["amenity"="nightclub"]({miny},{minx},{maxy},{maxx});
  node["amenity"="pub"]({miny},{minx},{maxy},{maxx});
  node["amenity"="bar"]({miny},{minx},{maxy},{maxx});
  node["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  node["leisure"="adult_gaming_centre"]({miny},{minx},{maxy},{maxx});
  node["amenity"="casino"]({miny},{minx},{maxy},{maxx});
  node["amenity"="gambling"]({miny},{minx},{maxy},{maxx});
  node["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  node["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  node["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  node["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  node["leisure"="escape_game"]({miny},{minx},{maxy},{maxx});
  node["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  node["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  node["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  node["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  node["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  node["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  node["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  node["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  node["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  way["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  way["social_facility"="social_club"]({miny},{minx},{maxy},{maxx});
  way["amenity"="nightclub"]({miny},{minx},{maxy},{maxx});
  way["amenity"="pub"]({miny},{minx},{maxy},{maxx});
  way["amenity"="bar"]({miny},{minx},{maxy},{maxx});
  way["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  way["leisure"="adult_gaming_centre"]({miny},{minx},{maxy},{maxx});
  way["amenity"="casino"]({miny},{minx},{maxy},{maxx});
  way["amenity"="gambling"]({miny},{minx},{maxy},{maxx});
  way["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  way["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  way["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  way["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  way["leisure"="escape_game"]({miny},{minx},{maxy},{maxx});
  way["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  way["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  way["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  way["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  way["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  way["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  way["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  way["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  way["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  relation["social_facility"="social_club"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="nightclub"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="pub"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="bar"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="adult_gaming_centre"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="casino"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="gambling"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="escape_game"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="park"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "leisure": node.tags.get("leisure"),
        "tourism": node.tags.get("tourism"),
        "amenity": node.tags.get("amenity"),
        "social_fac": node.tags.get("social_facility"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "leisure": way.tags.get("leisure"),
            "tourism": way.tags.get("tourism"),
            "amenity": way.tags.get("amenity"),
            "social_fac": way.tags.get("social_facility"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "leisure": relation.tags.get("leisure"),
            "tourism": relation.tags.get("tourism"),
            "amenity": relation.tags.get("amenity"),
            "social_fac": relation.tags.get("social_facility"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "En2_2"

# Save as a Shapefile
layer_name = "En2_2.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")


**En2_3 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
  node["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  node["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  node["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  node["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  node["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  node["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  node["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  node["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  node["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  node["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  node["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  node["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  node["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  node["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  node["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  node["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  way["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  way["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  way["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  way["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  way["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  way["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  way["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  way["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  way["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  way["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  way["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  way["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  way["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  way["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  way["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  way["leisure"="park"]({miny},{minx},{maxy},{maxx});
  
  relation["amenity"="cafe"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="zoo"]({miny},{minx},{maxy},{maxx});
  relation["amenity"="cinema"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="outdoor_seating"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="aquarium"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="theme_park"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="water_park"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="resort"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="beach_resort"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="swimming_area"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="dog_park"]({miny},{minx},{maxy},{maxx});
  relation["tourism"="picnic_site"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="picnic_table"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="firepit"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="amusement_arcade"]({miny},{minx},{maxy},{maxx});
  relation["leisure"="park"]({miny},{minx},{maxy},{maxx});
);
out center;
"""

# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "leisure": node.tags.get("leisure"),
        "tourism": node.tags.get("tourism"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "leisure": way.tags.get("leisure"),
            "tourism": way.tags.get("tourism"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

for relation in result.relations:
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "leisure": relation.tags.get("leisure"),
            "tourism": relation.tags.get("tourism"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "En2_3"

# Save as a Shapefile
layer_name = "En2_3.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")


**En3 Category**

In [ ]:
# Build Overpass query for the bounding box (out center for polygons)
query = f"""
[out:xml] [timeout:300];
(
node["leisure"="golf_course"]({miny},{minx},{maxy},{maxx});
node["golf"="clubhuse"]({miny},{minx},{maxy},{maxx});
node["golf"="tee"]({miny},{minx},{maxy},{maxx});
node["golf"="hole"]({miny},{minx},{maxy},{maxx});
node["golf"="pin"]({miny},{minx},{maxy},{maxx});
node["golf"="bunker"]({miny},{minx},{maxy},{maxx});
node["golf"="water_hazard"]({miny},{minx},{maxy},{maxx});
node["natural"="water"]({miny},{minx},{maxy},{maxx});
node["golf"="lateral_water_hazard"]({miny},{minx},{maxy},{maxx});
node["golf"="green"]({miny},{minx},{maxy},{maxx});
node["golf"="fairway"]({miny},{minx},{maxy},{maxx});
node["surface"="grass"]({miny},{minx},{maxy},{maxx});
node["golf"="rough"]({miny},{minx},{maxy},{maxx});
node["golf"="driving_range"]({miny},{minx},{maxy},{maxx});
node["leisure"="stadium"]({miny},{minx},{maxy},{maxx});
node["leisure"="sports_centre"]({miny},{minx},{maxy},{maxx});
node["leisure"="fitness_centre"]({miny},{minx},{maxy},{maxx});
node["leisure"="fitness_station"]({miny},{minx},{maxy},{maxx});
node["amenity"="dojo"]({miny},{minx},{maxy},{maxx});
node["leisure"="pitch"]({miny},{minx},{maxy},{maxx});
node["leisure"="track"]({miny},{minx},{maxy},{maxx});
node["amenity"="dive_centre"]({miny},{minx},{maxy},{maxx});
node["leisure"="ice_rink"]({miny},{minx},{maxy},{maxx});
node["leisure"="bowling_alley"]({miny},{minx},{maxy},{maxx});
node["leisure"="miniature_golf"]({miny},{minx},{maxy},{maxx});
node["sport"="multi"]({miny},{minx},{maxy},{maxx});
node["sport"="9pin"]({miny},{minx},{maxy},{maxx});
node["sport"="10pin"]({miny},{minx},{maxy},{maxx});
node["sport"="archery"]({miny},{minx},{maxy},{maxx});
node["sport"="athletics"]({miny},{minx},{maxy},{maxx});
node["sport"="running"]({miny},{minx},{maxy},{maxx});
node["sport"="climbing"]({miny},{minx},{maxy},{maxx});
node["climbing"="route"]({miny},{minx},{maxy},{maxx});
node["sport"="canoe"]({miny},{minx},{maxy},{maxx});
node["sport"="rowing"]({miny},{minx},{maxy},{maxx});
node["sport"="cycling"]({miny},{minx},{maxy},{maxx});
node["sport"="dog_racing"]({miny},{minx},{maxy},{maxx});
node["sport"="equestrian"]({miny},{minx},{maxy},{maxx});
node["sport"="horse_racing"]({miny},{minx},{maxy},{maxx});
node["sport"="gymnastics"]({miny},{minx},{maxy},{maxx});
node["sport"="skating"]({miny},{minx},{maxy},{maxx});
node["sport"="ice_skating"]({miny},{minx},{maxy},{maxx});
node["sport"="curling"]({miny},{minx},{maxy},{maxx});
node["sport"="roller_skating"]({miny},{minx},{maxy},{maxx});
node["sport"="skateboard"]({miny},{minx},{maxy},{maxx});
node["sport"="swimming"]({miny},{minx},{maxy},{maxx});
node["sport"="scuba_diving"]({miny},{minx},{maxy},{maxx});
node["sport"="skiing"]({miny},{minx},{maxy},{maxx});
node["sport"="shooting"]({miny},{minx},{maxy},{maxx});
node["sport"="chess"]({miny},{minx},{maxy},{maxx});
node["sport"="soccer"]({miny},{minx},{maxy},{maxx});
node["sport"="australian_football"]({miny},{minx},{maxy},{maxx});
node["sport"="american_football"]({miny},{minx},{maxy},{maxx});
node["sport"="canadian_football"]({miny},{minx},{maxy},{maxx});
node["sport"="gaelic_games"]({miny},{minx},{maxy},{maxx});
node["sport"="rugby_league"]({miny},{minx},{maxy},{maxx});
node["sport"="rugby_union"]({miny},{minx},{maxy},{maxx});
node["sport"="baseball"]({miny},{minx},{maxy},{maxx});
node["sport"="basketball"]({miny},{minx},{maxy},{maxx});
node["sport"="handball"]({miny},{minx},{maxy},{maxx});
node["sport"="volleyball"]({miny},{minx},{maxy},{maxx});
node["sport"="beachvolleyball"]({miny},{minx},{maxy},{maxx});
node["sport"="billiards"]({miny},{minx},{maxy},{maxx});
node["sport"="golf"]({miny},{minx},{maxy},{maxx});
node["sport"="boules"]({miny},{minx},{maxy},{maxx});
node["sport"="bowls"]({miny},{minx},{maxy},{maxx});
node["sport"="cricket"]({miny},{minx},{maxy},{maxx});
node["sport"="croquet"]({miny},{minx},{maxy},{maxx});
node["sport"="field_hockey"]({miny},{minx},{maxy},{maxx});
node["sport"="ice_hockey"]({miny},{minx},{maxy},{maxx});
node["sport"="pelota"]({miny},{minx},{maxy},{maxx});
node["sport"="racquet"]({miny},{minx},{maxy},{maxx});
node["sport"="table_tennis"]({miny},{minx},{maxy},{maxx});
node["sport"="tennis"]({miny},{minx},{maxy},{maxx});
node["sport"="motor"]({miny},{minx},{maxy},{maxx});
node["sport"="karting"]({miny},{minx},{maxy},{maxx});
node["sport"="motocross"]({miny},{minx},{maxy},{maxx});
node["sport"="safety_training"]({miny},{minx},{maxy},{maxx});
node["sport"="model_aerodrome"]({miny},{minx},{maxy},{maxx});
node["sport"="rc_car"]({miny},{minx},{maxy},{maxx});
node["highway"="raceway"]({miny},{minx},{maxy},{maxx});

way["leisure"="golf_course"]({miny},{minx},{maxy},{maxx});
way["golf"="clubhuse"]({miny},{minx},{maxy},{maxx});
way["golf"="tee"]({miny},{minx},{maxy},{maxx});
way["golf"="hole"]({miny},{minx},{maxy},{maxx});
way["golf"="pin"]({miny},{minx},{maxy},{maxx});
way["golf"="bunker"]({miny},{minx},{maxy},{maxx});
way["golf"="water_hazard"]({miny},{minx},{maxy},{maxx});
way["natural"="water"]({miny},{minx},{maxy},{maxx});
way["golf"="lateral_water_hazard"]({miny},{minx},{maxy},{maxx});
way["golf"="green"]({miny},{minx},{maxy},{maxx});
way["golf"="fairway"]({miny},{minx},{maxy},{maxx});
way["surface"="grass"]({miny},{minx},{maxy},{maxx});
way["golf"="rough"]({miny},{minx},{maxy},{maxx});
way["golf"="driving_range"]({miny},{minx},{maxy},{maxx});
way["leisure"="stadium"]({miny},{minx},{maxy},{maxx});
way["leisure"="sports_centre"]({miny},{minx},{maxy},{maxx});
way["leisure"="fitness_centre"]({miny},{minx},{maxy},{maxx});
way["leisure"="fitness_station"]({miny},{minx},{maxy},{maxx});
way["amenity"="dojo"]({miny},{minx},{maxy},{maxx});
way["leisure"="pitch"]({miny},{minx},{maxy},{maxx});
way["leisure"="track"]({miny},{minx},{maxy},{maxx});
way["amenity"="dive_centre"]({miny},{minx},{maxy},{maxx});
way["leisure"="ice_rink"]({miny},{minx},{maxy},{maxx});
way["leisure"="bowling_alley"]({miny},{minx},{maxy},{maxx});
way["leisure"="miniature_golf"]({miny},{minx},{maxy},{maxx});
way["sport"="multi"]({miny},{minx},{maxy},{maxx});
way["sport"="9pin"]({miny},{minx},{maxy},{maxx});
way["sport"="10pin"]({miny},{minx},{maxy},{maxx});
way["sport"="archery"]({miny},{minx},{maxy},{maxx});
way["sport"="athletics"]({miny},{minx},{maxy},{maxx});
way["sport"="running"]({miny},{minx},{maxy},{maxx});
way["sport"="climbing"]({miny},{minx},{maxy},{maxx});
way["climbing"="route"]({miny},{minx},{maxy},{maxx});
way["sport"="canoe"]({miny},{minx},{maxy},{maxx});
way["sport"="rowing"]({miny},{minx},{maxy},{maxx});
way["sport"="cycling"]({miny},{minx},{maxy},{maxx});
way["sport"="dog_racing"]({miny},{minx},{maxy},{maxx});
way["sport"="equestrian"]({miny},{minx},{maxy},{maxx});
way["sport"="horse_racing"]({miny},{minx},{maxy},{maxx});
way["sport"="gymnastics"]({miny},{minx},{maxy},{maxx});
way["sport"="skating"]({miny},{minx},{maxy},{maxx});
way["sport"="ice_skating"]({miny},{minx},{maxy},{maxx});
way["sport"="curling"]({miny},{minx},{maxy},{maxx});
way["sport"="roller_skating"]({miny},{minx},{maxy},{maxx});
way["sport"="skateboard"]({miny},{minx},{maxy},{maxx});
way["sport"="swimming"]({miny},{minx},{maxy},{maxx});
way["sport"="scuba_diving"]({miny},{minx},{maxy},{maxx});
way["sport"="skiing"]({miny},{minx},{maxy},{maxx});
way["sport"="shooting"]({miny},{minx},{maxy},{maxx});
way["sport"="chess"]({miny},{minx},{maxy},{maxx});
way["sport"="soccer"]({miny},{minx},{maxy},{maxx});
way["sport"="australian_football"]({miny},{minx},{maxy},{maxx});
way["sport"="american_football"]({miny},{minx},{maxy},{maxx});
way["sport"="canadian_football"]({miny},{minx},{maxy},{maxx});
way["sport"="gaelic_games"]({miny},{minx},{maxy},{maxx});
way["sport"="rugby_league"]({miny},{minx},{maxy},{maxx});
way["sport"="rugby_union"]({miny},{minx},{maxy},{maxx});
way["sport"="baseball"]({miny},{minx},{maxy},{maxx});
way["sport"="basketball"]({miny},{minx},{maxy},{maxx});
way["sport"="handball"]({miny},{minx},{maxy},{maxx});
way["sport"="volleyball"]({miny},{minx},{maxy},{maxx});
way["sport"="beachvolleyball"]({miny},{minx},{maxy},{maxx});
way["sport"="billiards"]({miny},{minx},{maxy},{maxx});
way["sport"="golf"]({miny},{minx},{maxy},{maxx});
way["sport"="boules"]({miny},{minx},{maxy},{maxx});
way["sport"="bowls"]({miny},{minx},{maxy},{maxx});
way["sport"="cricket"]({miny},{minx},{maxy},{maxx});
way["sport"="croquet"]({miny},{minx},{maxy},{maxx});
way["sport"="field_hockey"]({miny},{minx},{maxy},{maxx});
way["sport"="ice_hockey"]({miny},{minx},{maxy},{maxx});
way["sport"="pelota"]({miny},{minx},{maxy},{maxx});
way["sport"="racquet"]({miny},{minx},{maxy},{maxx});
way["sport"="table_tennis"]({miny},{minx},{maxy},{maxx});
way["sport"="tennis"]({miny},{minx},{maxy},{maxx});
way["sport"="motor"]({miny},{minx},{maxy},{maxx});
way["sport"="karting"]({miny},{minx},{maxy},{maxx});
way["sport"="motocross"]({miny},{minx},{maxy},{maxx});
way["sport"="safety_training"]({miny},{minx},{maxy},{maxx});
way["sport"="model_aerodrome"]({miny},{minx},{maxy},{maxx});
way["sport"="rc_car"]({miny},{minx},{maxy},{maxx});
way["highway"="raceway"]({miny},{minx},{maxy},{maxx});

relation["leisure"="golf_course"]({miny},{minx},{maxy},{maxx});
relation["golf"="clubhuse"]({miny},{minx},{maxy},{maxx});
relation["golf"="tee"]({miny},{minx},{maxy},{maxx});
relation["golf"="hole"]({miny},{minx},{maxy},{maxx});
relation["golf"="pin"]({miny},{minx},{maxy},{maxx});
relation["golf"="bunker"]({miny},{minx},{maxy},{maxx});
relation["golf"="water_hazard"]({miny},{minx},{maxy},{maxx});
relation["natural"="water"]({miny},{minx},{maxy},{maxx});
relation["golf"="lateral_water_hazard"]({miny},{minx},{maxy},{maxx});
relation["golf"="green"]({miny},{minx},{maxy},{maxx});
relation["golf"="fairway"]({miny},{minx},{maxy},{maxx});
relation["surface"="grass"]({miny},{minx},{maxy},{maxx});
relation["golf"="rough"]({miny},{minx},{maxy},{maxx});
relation["golf"="driving_range"]({miny},{minx},{maxy},{maxx});
relation["leisure"="stadium"]({miny},{minx},{maxy},{maxx});
relation["leisure"="sports_centre"]({miny},{minx},{maxy},{maxx});
relation["leisure"="fitness_centre"]({miny},{minx},{maxy},{maxx});
relation["leisure"="fitness_station"]({miny},{minx},{maxy},{maxx});
relation["amenity"="dojo"]({miny},{minx},{maxy},{maxx});
relation["leisure"="pitch"]({miny},{minx},{maxy},{maxx});
relation["leisure"="track"]({miny},{minx},{maxy},{maxx});
relation["amenity"="dive_centre"]({miny},{minx},{maxy},{maxx});
relation["leisure"="ice_rink"]({miny},{minx},{maxy},{maxx});
relation["leisure"="bowling_alley"]({miny},{minx},{maxy},{maxx});
relation["leisure"="miniature_golf"]({miny},{minx},{maxy},{maxx});
relation["sport"="multi"]({miny},{minx},{maxy},{maxx});
relation["sport"="9pin"]({miny},{minx},{maxy},{maxx});
relation["sport"="10pin"]({miny},{minx},{maxy},{maxx});
relation["sport"="archery"]({miny},{minx},{maxy},{maxx});
relation["sport"="athletics"]({miny},{minx},{maxy},{maxx});
relation["sport"="running"]({miny},{minx},{maxy},{maxx});
relation["sport"="climbing"]({miny},{minx},{maxy},{maxx});
relation["climbing"="route"]({miny},{minx},{maxy},{maxx});
relation["sport"="canoe"]({miny},{minx},{maxy},{maxx});
relation["sport"="rowing"]({miny},{minx},{maxy},{maxx});
relation["sport"="cycling"]({miny},{minx},{maxy},{maxx});
relation["sport"="dog_racing"]({miny},{minx},{maxy},{maxx});
relation["sport"="equestrian"]({miny},{minx},{maxy},{maxx});
relation["sport"="horse_racing"]({miny},{minx},{maxy},{maxx});
relation["sport"="gymnastics"]({miny},{minx},{maxy},{maxx});
relation["sport"="skating"]({miny},{minx},{maxy},{maxx});
relation["sport"="ice_skating"]({miny},{minx},{maxy},{maxx});
relation["sport"="curling"]({miny},{minx},{maxy},{maxx});
relation["sport"="roller_skating"]({miny},{minx},{maxy},{maxx});
relation["sport"="skateboard"]({miny},{minx},{maxy},{maxx});
relation["sport"="swimming"]({miny},{minx},{maxy},{maxx});
relation["sport"="scuba_diving"]({miny},{minx},{maxy},{maxx});
relation["sport"="skiing"]({miny},{minx},{maxy},{maxx});
relation["sport"="shooting"]({miny},{minx},{maxy},{maxx});
relation["sport"="chess"]({miny},{minx},{maxy},{maxx});
relation["sport"="soccer"]({miny},{minx},{maxy},{maxx});
relation["sport"="australian_football"]({miny},{minx},{maxy},{maxx});
relation["sport"="american_football"]({miny},{minx},{maxy},{maxx});
relation["sport"="canadian_football"]({miny},{minx},{maxy},{maxx});
relation["sport"="gaelic_games"]({miny},{minx},{maxy},{maxx});
relation["sport"="rugby_league"]({miny},{minx},{maxy},{maxx});
relation["sport"="rugby_union"]({miny},{minx},{maxy},{maxx});
relation["sport"="baseball"]({miny},{minx},{maxy},{maxx});
relation["sport"="basketball"]({miny},{minx},{maxy},{maxx});
relation["sport"="handball"]({miny},{minx},{maxy},{maxx});
relation["sport"="volleyball"]({miny},{minx},{maxy},{maxx});
relation["sport"="beachvolleyball"]({miny},{minx},{maxy},{maxx});
relation["sport"="billiards"]({miny},{minx},{maxy},{maxx});
relation["sport"="golf"]({miny},{minx},{maxy},{maxx});
relation["sport"="boules"]({miny},{minx},{maxy},{maxx});
relation["sport"="bowls"]({miny},{minx},{maxy},{maxx});
relation["sport"="cricket"]({miny},{minx},{maxy},{maxx});
relation["sport"="croquet"]({miny},{minx},{maxy},{maxx});
relation["sport"="field_hockey"]({miny},{minx},{maxy},{maxx});
relation["sport"="ice_hockey"]({miny},{minx},{maxy},{maxx});
relation["sport"="pelota"]({miny},{minx},{maxy},{maxx});
relation["sport"="racquet"]({miny},{minx},{maxy},{maxx});
relation["sport"="table_tennis"]({miny},{minx},{maxy},{maxx});
relation["sport"="tennis"]({miny},{minx},{maxy},{maxx});
relation["sport"="motor"]({miny},{minx},{maxy},{maxx});
relation["sport"="karting"]({miny},{minx},{maxy},{maxx});
relation["sport"="motocross"]({miny},{minx},{maxy},{maxx});
relation["sport"="safety_training"]({miny},{minx},{maxy},{maxx});
relation["sport"="model_aerodrome"]({miny},{minx},{maxy},{maxx});
relation["sport"="rc_car"]({miny},{minx},{maxy},{maxx});
relation["highway"="raceway"]({miny},{minx},{maxy},{maxx});
);
out center;
"""


# Connect to Overpass API and run the query
api = overpy.Overpass()
result = api.query(query)

# Parse results and convert to a GeoDataFrame
data = []
for node in result.nodes:
    data.append({
        "id": node.id,
        "leisure": node.tags.get("leisure"),
        "golf": node.tags.get("golf"),
        "sport": node.tags.get("sport"),
        "natural": node.tags.get("natural"),
        "surface": node.tags.get("surface"),
        "highway": node.tags.get("highway"),
        "amenity": node.tags.get("amenity"),
        "name": node.tags.get("name"),
        "geometry": Point(float(node.lon), float(node.lat))
    })

for way in result.ways:
    # Use the center point for ways
    if way.center_lat is not None and way.center_lon is not None:
        data.append({
            "id": way.id,
            "leisure": way.tags.get("leisure"),
            "golf": way.tags.get("golf"),
            "sport": way.tags.get("sport"),
            "natural": way.tags.get("natural"),
            "surface": way.tags.get("surface"),
            "highway": way.tags.get("highway"),
            "amenity": way.tags.get("amenity"),
            "name": way.tags.get("name"),
            "geometry": Point(float(way.center_lon), float(way.center_lat))
        })

    
for relation in result.relations:
    # Use the center point for relations
    if relation.center_lat is not None and relation.center_lon is not None:
        data.append({
            "id": relation.id,
            "leisure": relation.tags.get("leisure"),
            "golf": relation.tags.get("golf"),
            "sport": relation.tags.get("sport"),
            "natural": relation.tags.get("natural"),
            "surface": relation.tags.get("surface"),
            "highway": relation.tags.get("highway"),
            "amenity": relation.tags.get("amenity"),
            "name": relation.tags.get("name"),
            "geometry": Point(float(relation.center_lon), float(relation.center_lat))
        })

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")

# Add Category column
gdf["Category"] = "En3"

# Save as a Shapefile
layer_name = "En3.shp"
output_path = output_folder / layer_name
gdf.to_file(output_path, driver="ESRI Shapefile")

print("Shapefile saved successfully.")

In [ ]:
print("All POI categories exported successfully.")